In [12]:
import os
from dotenv import load_dotenv
from langsmith import wrappers
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langsmith import traceable
from langsmith import Client
from typing_extensions import Annotated,TypedDict
from google import genai
from google.genai import types

load_dotenv()

True

In [29]:
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [6]:
# models/gemini-2.5-flash
# models/gemini-2.5-pro
# models/gemini-2.5-flash-preview-tts
# models/gemini-2.5-pro-preview-tts
# models/gemma-4-26b-a4b-it
# models/gemma-4-31b-it
# models/gemini-flash-latest
# models/gemini-flash-lite-latest
# models/gemini-pro-latest
# models/gemini-2.5-flash-lite
# models/gemini-2.5-flash-image
# models/gemini-3-flash-preview
# models/gemini-3.1-pro-preview
# models/gemini-3.1-pro-preview-customtools
# models/gemini-3.1-flash-lite-preview
# models/gemini-3.1-flash-lite
# models/gemini-3-pro-image-preview
# models/gemini-3-pro-image
# models/nano-banana-pro-preview
# models/gemini-3.1-flash-image-preview
# models/gemini-3.1-flash-image
# models/gemini-3.1-flash-lite-image
# models/gemini-3.5-flash
# models/gemini-3.5-flash-lite
# models/gemini-omni-flash-preview
# models/gemini-3.6-flash
# models/gemini-3.7-flash
# models/lyria-3-clip-preview
# models/lyria-3-pro-preview
# models/gemini-3.1-flash-tts-preview
# models/gemini-robotics-er-1.6-preview
# models/gemini-robotics-er-2-preview
# models/gemini-2.5-computer-use-preview-10-2025
# models/antigravity-preview-05-2026
# models/deep-research-max-preview-04-2026
# models/deep-research-preview-04-2026
# models/deep-research-pro-preview-12-2025
# models/gemini-embedding-001
# models/gemini-embedding-2-preview
# models/gemini-embedding-2
# models/aqa
# models/veo-3.1-generate-preview
# models/veo-3.1-fast-generate-preview
# models/veo-3.1-lite-generate-preview
# models/gemini-2.5-flash-native-audio-latest
# models/gemini-2.5-flash-native-audio-preview-09-2025
# models/gemini-2.5-flash-native-audio-preview-12-2025
# models/gemini-3.1-flash-live-preview
# models/gemini-robotics-er-2-streaming-preview
# models/gemini-3.5-live-translate-preview
# models/lyria-realtime-exp

In [7]:
client = Client()

# define the dataset
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['f6522a16-af48-4698-bffe-a4a9562c6488',
  '1ac7a5d3-135c-4ac6-a606-8973c9b09636',
  'de35cfb8-dedc-4d58-829d-02b3bbbf0f0b',
  '98a4e425-7fb9-4bdc-94f1-75513da3a54f',
  'd84e14df-be35-40be-a5c0-82d1a168dbff'],
 'count': 5,
 'as_of': '2026-08-17T13:03:33.4423972Z'}

### LLM as a judge

In [35]:
genai_client = genai.Client()

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:    {inputs['question']}    
    Here is the real answer:    {reference_outputs['answer']}    
    You are grading the following predicted answer:    {outputs['response']}    
    Respond with CORRECT or INCORRECT:    Grade:    """

    full_prompt = f"System Instructions:\n{eval_instructions}\n\nTask:\n{user_content}"

    response = genai_client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = full_prompt,
        config = {"temperature": 0}
    )

    return "CORRECT" in response.text.strip().upper()


In [36]:
# Concisions- checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs:dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [37]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model:str = "gemini-3.5-flash-lite", instructions:str = default_instructions)-> str:

    genai_client = genai.Client()

    response = genai_client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0
       ),
    )
    return response.text

In [38]:
# calling my app for datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [39]:
# evaluations
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-flash-chatbot"
)

View the evaluation results for experiment: 'gemini-flash-chatbot-c989ec50' at:
https://smith.langchain.com/o/2722c776-1db7-404e-b37b-fd36cd1fb1be/datasets/5ea671a1-7943-49e4-a7ed-9eba482e236a/compare?selectedSessions=2162b5e6-0815-4b6b-a6d5-9e029078801d




5it [00:10,  2.01s/it]
